# TerraScope — Satellite Change Intelligence Platform
## Complete Google Colab GPU Model Training & Benchmarking Pipeline

This notebook trains and quantitatively benchmarks three deep learning architectures for bi-temporal satellite change detection:
1. **Siamese U-Net** (Convolutional Neural Network baseline)
2. **Siamese Vision Transformer (ViT)** (Pure self-attention backbone)
3. **Siamese Swin Transformer** (Hierarchical shifted-window vision transformer)

### Benchmark Datasets:
- **LEVIR-CD**: Ultra-high resolution bi-temporal optical building change detection (RGB).
- **OSCD**: Onera Satellite Change Detection multi-spectral Sentinel-2 dataset.

---
### Workflow Overview:
1. **Environment Setup & GPU Verification** (CUDA, PyTorch, Dependencies)
2. **Mount Google Drive & Configure Dataset Paths**
3. **Pre-flight Dataset Verification**
4. **Quick Smoke Test (1-epoch pre-flight validation)**
5. **Systematic Model Training (LEVIR-CD & OSCD)**
6. **Quantitative Benchmarking & Evaluation on Test Splits**
7. **Export Visualizations, Confusion Matrices, Prediction Overlays & Checkpoints to Google Drive**


### Step 1: Mount Google Drive
Mount your Google Drive to access your uploaded repository code and datasets, and to automatically save trained model checkpoints.


In [ ]:
from google.colab import drive
import os
import sys
from pathlib import Path

drive.mount("/content/drive")
print("Google Drive successfully mounted at /content/drive")


### Step 2: GPU Accelerator Verification
Verify that a CUDA-capable GPU (T4, V100, or A100) is assigned to this Colab session.


In [ ]:
!nvidia-smi

import torch
print("--- PyTorch Compute Environment ---")
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Device Memory   : {total_mem_gb:.2f} GB")
else:
    raise SystemError("CUDA GPU not detected! Go to Runtime > Change runtime type > select T4 GPU.")


### Step 3: Locate or Copy TerraScope Repository
Set the workspace path where `satellite` is stored in your Google Drive (e.g. `/content/drive/MyDrive/satellite`).
To maximize I/O throughput during training, we can copy the project and datasets to local `/content/` disk.


In [ ]:
# Define Google Drive paths
DRIVE_PROJECT_PATH = Path("/content/drive/MyDrive/satellite")
LOCAL_PROJECT_PATH = Path("/content/satellite")

# If project is on Drive, copy to local /content for high-speed SSD access
if DRIVE_PROJECT_PATH.exists():
    if not LOCAL_PROJECT_PATH.exists():
        print(f"Copying project from {DRIVE_PROJECT_PATH} to {LOCAL_PROJECT_PATH} for faster I/O...")
        !cp -r "$DRIVE_PROJECT_PATH" /content/satellite
    os.chdir(str(LOCAL_PROJECT_PATH))
    PROJECT_ROOT = LOCAL_PROJECT_PATH
elif LOCAL_PROJECT_PATH.exists():
    os.chdir(str(LOCAL_PROJECT_PATH))
    PROJECT_ROOT = LOCAL_PROJECT_PATH
else:
    PROJECT_ROOT = Path(os.getcwd())

print(f"Active Working Directory: {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


### Step 4: Install Dependencies
Install all required libraries (`rasterio`, `timm`, `geopandas`, etc.).


In [ ]:
!pip install -q rasterio geopandas timm reportlab python-dotenv

# Verify key imports
import rasterio
import timm
from ml.models import get_model
from ml.utils.device import detect_device

print(f"Active Device detected: {detect_device()}")
print("All dependencies loaded successfully.")


### Step 5: Configure Dataset Paths
Specify the locations of your LEVIR-CD and OSCD datasets.
You can point directly to folders in Google Drive (`/content/drive/MyDrive/...`) or local disk (`/content/data/...`).


In [ ]:
# Define dataset paths - UPDATE THESE TO MATCH YOUR GOOGLE DRIVE STRUCTURE
LEVIR_CD_DIR = "/content/drive/MyDrive/data/LEVIR-CD"
OSCD_DIR = "/content/drive/MyDrive/data/onera DATA/Onera Satellite Change Detection dataset - Images"

# Check existence
print("LEVIR-CD Path:", LEVIR_CD_DIR, "-> Exists:", Path(LEVIR_CD_DIR).exists())
print("OSCD Path    :", OSCD_DIR, "-> Exists:", Path(OSCD_DIR).exists())


### Step 6: Dataset Split Verification
Run `prepare_dataset.py` to ensure all train, validation, and test splits and label files resolve properly without dimension mismatches.


In [ ]:
# Verify LEVIR-CD if present
if Path(LEVIR_CD_DIR).exists():
    print("--- Verifying LEVIR-CD ---")
    !python scripts/prepare_dataset.py --dataset levir --source-dir "$LEVIR_CD_DIR"

# Verify OSCD
if Path(OSCD_DIR).exists():
    print("--- Verifying OSCD ---")
    !python scripts/prepare_dataset.py --dataset oscd --source-dir "$OSCD_DIR" --bands B04 B03 B02


### Step 7: Fast Pre-flight Smoke Test (1-epoch validation)
Before initiating multi-epoch training, run a 1-epoch smoke test with a small batch to confirm that CUDA forward pass, loss calculation, backpropagation, and checkpoint saving execute without issues.


In [ ]:
print("=== RUNNING PRE-FLIGHT SMOKE TEST ===")
!python scripts/train_model.py \
    --model siamese_unet \
    --dataset OSCD \
    --data-dir "$OSCD_DIR" \
    --epochs 1 \
    --batch-size 4 \
    --device cuda \
    --seed 42

print("Smoke test completed successfully! Ready for full training.")


## Step 8: Systematic Model Training Pipeline

We train the three architectures under identical hyperparameters:
- **Optimizer**: AdamW (`--optimizer adamw`)
- **Loss**: Binary Cross-Entropy + Dice Loss (`--loss bce_dice`)
- **Scheduler**: Cosine Annealing (`--scheduler cosine`)
- **Learning Rate**: $3 \times 10^{-4}$
- **Batch Size**: 8 (balanced for GPU VRAM and gradient stability)
- **Patch Size**: 256
- **Seed**: 42 (strict reproducibility across all runs)

---
### Part A: Train on LEVIR-CD Dataset


In [ ]:
# --- EXPERIMENT 1: Siamese U-Net on LEVIR-CD ---
print("=== Training Experiment 1/6: Siamese U-Net on LEVIR-CD ===")
!python scripts/train_model.py \
    --model siamese_unet \
    --dataset LEVIR-CD \
    --data-dir "$LEVIR_CD_DIR" \
    --epochs 20 \
    --batch-size 8 \
    --patch-size 256 \
    --lr 3e-4 \
    --device cuda \
    --seed 42

# --- EXPERIMENT 2: Siamese ViT on LEVIR-CD ---
print("=== Training Experiment 2/6: Siamese ViT on LEVIR-CD ===")
!python scripts/train_model.py \
    --model vit \
    --dataset LEVIR-CD \
    --data-dir "$LEVIR_CD_DIR" \
    --epochs 20 \
    --batch-size 8 \
    --patch-size 256 \
    --lr 3e-4 \
    --device cuda \
    --seed 42

# --- EXPERIMENT 3: Siamese Swin Transformer on LEVIR-CD ---
print("=== Training Experiment 3/6: Siamese Swin Transformer on LEVIR-CD ===")
!python scripts/train_model.py \
    --model swin \
    --dataset LEVIR-CD \
    --data-dir "$LEVIR_CD_DIR" \
    --epochs 20 \
    --batch-size 8 \
    --patch-size 256 \
    --lr 3e-4 \
    --device cuda \
    --seed 42


### Part B: Train on OSCD (Sentinel-2 Multi-spectral) Dataset


In [ ]:
# --- EXPERIMENT 4: Siamese U-Net on OSCD ---
print("=== Training Experiment 4/6: Siamese U-Net on OSCD ===")
!python scripts/train_model.py \
    --model siamese_unet \
    --dataset OSCD \
    --data-dir "$OSCD_DIR" \
    --epochs 25 \
    --batch-size 8 \
    --patch-size 256 \
    --bands B04 B03 B02 \
    --lr 3e-4 \
    --device cuda \
    --seed 42

# --- EXPERIMENT 5: Siamese ViT on OSCD ---
print("=== Training Experiment 5/6: Siamese ViT on OSCD ===")
!python scripts/train_model.py \
    --model vit \
    --dataset OSCD \
    --data-dir "$OSCD_DIR" \
    --epochs 25 \
    --batch-size 8 \
    --patch-size 256 \
    --bands B04 B03 B02 \
    --lr 3e-4 \
    --device cuda \
    --seed 42

# --- EXPERIMENT 6: Siamese Swin Transformer on OSCD ---
print("=== Training Experiment 6/6: Siamese Swin Transformer on OSCD ===")
!python scripts/train_model.py \
    --model swin \
    --dataset OSCD \
    --data-dir "$OSCD_DIR" \
    --epochs 25 \
    --batch-size 8 \
    --patch-size 256 \
    --bands B04 B03 B02 \
    --lr 3e-4 \
    --device cuda \
    --seed 42


### Checkpoint Resumption (If Session Disconnects)
If your Colab instance times out or disconnects during training, you can resume immediately from the latest checkpoint by appending `--resume models/checkpoints/{model}_{dataset}_latest.pt`:

```bash
!python scripts/train_model.py \
    --model vit \
    --dataset LEVIR-CD \
    --data-dir "$LEVIR_CD_DIR" \
    --resume models/checkpoints/vit_levir-cd_latest.pt \
    --device cuda
```


## Step 9: Standardized Quantitative Benchmarking on Test Split
Evaluate all three architectures simultaneously under identical test sets, thresholds, and metrics.
This generates:
- JSON & CSV benchmark reports (`experiments/results/`)
- Comparison bar charts (`outputs/visualizations/`)
- Confusion matrices for each architecture
- Qualitative visual prediction tiles


In [ ]:
print("=== Benchmarking Models on LEVIR-CD Test Set ===")
!python scripts/evaluate_model.py \
    --dataset LEVIR-CD \
    --data-dir "$LEVIR_CD_DIR" \
    --split test \
    --patch-size 256 \
    --batch-size 8 \
    --threshold 0.5 \
    --device cuda \
    --save-visuals \
    --visual-samples 6


In [ ]:
print("=== Benchmarking Models on OSCD Test Set ===")
!python scripts/evaluate_model.py \
    --dataset OSCD \
    --data-dir "$OSCD_DIR" \
    --split test \
    --patch-size 256 \
    --batch-size 8 \
    --threshold 0.5 \
    --device cuda \
    --bands B04 B03 B02 \
    --save-visuals \
    --visual-samples 6


## Step 10: Display Evaluation Results & Visualizations


In [ ]:
import glob
from IPython.display import Image, display

# Display generated comparison charts
charts = sorted(glob.glob("outputs/visualizations/*_comparison.png"))
for c in charts:
    print(f"Displaying: {c}")
    display(Image(filename=c))

# Display confusion matrices
cms = sorted(glob.glob("outputs/visualizations/*_cm.png"))
for cm in cms:
    print(f"Displaying: {cm}")
    display(Image(filename=cm))


## Step 11: Sync Checkpoints, Metrics & Visualizations to Google Drive
Permanently archive your trained weights (`*.pt`), experiment logs (`history.csv`, `config.json`), and visualization figures to a dedicated folder in Google Drive.


In [ ]:
DEST_BACKUP_DIR = "/content/drive/MyDrive/TerraScope_Outputs"
Path(DEST_BACKUP_DIR).mkdir(parents=True, exist_ok=True)

print(f"Archiving outputs to: {DEST_BACKUP_DIR} ...")
!cp -r models/checkpoints "$DEST_BACKUP_DIR/"
!cp -r experiments "$DEST_BACKUP_DIR/"
!cp -r outputs/visualizations "$DEST_BACKUP_DIR/"

print("All checkpoints, experiment results, and visualizations successfully backed up to Google Drive!")
!ls -la "$DEST_BACKUP_DIR/checkpoints"
!ls -la "$DEST_BACKUP_DIR/visualizations"
